# 🎯 Masterclass 02: Supervised Classification with Regularized Log-Loss
This notebook details probability classification pipelines:

1. **Project 1 (Theory Scratch)**: A custom binary Logistic Regression classifier with log-loss gradient descent.
2. **Project 2 (Applied Industry)**: A complete credit default pipeline solving heavy class imbalance with SMOTE and precision-recall curve boundary tuning.


## 📐 Part 1: Mathematical Foundations & LaTeX Proofs
To perform classification, we map linear inputs to a probability bounds space $[0,1]$ using the **Logistic Sigmoid Function**:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

### 1. Loss Formulation: Binary Cross-Entropy (Log Loss)
Instead of MSE (which yields a non-convex space on sigmoid outputs), we maximize the likelihood of correct classifications using log loss:
$$\mathcal{L}(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(h_\theta(x^{(i)})) + (1-y^{(i)}) \log(1-h_\theta(x^{(i)})) \right]$$
The gradient update rule is cleanly derived via the chain rule:
$$\frac{\partial \mathcal{L}}{\partial \theta_j} = \frac{1}{m} \sum_{i=1}^{m} (\sigma(\theta^T x^{(i)}) - y^{(i)}) x_j^{(i)}$$


## 🧠 Project 1: Vectorized Classifier from Scratch


In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.05, epochs=1000, reg_strength=0.1):
        self.lr = lr
        self.epochs = epochs
        self.reg = reg_strength
        self.w = None
        self.b = None

    def _sigmoid(self, z):
        z_clipped = np.clip(z, -25.0, 25.0)
        return 1.0 / (1.0 + np.exp(-z_clipped))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for epoch in range(self.epochs):
            z = np.dot(X, self.w) + self.b
            p = self._sigmoid(z)

            dw = (1 / n_samples) * np.dot(X.T, (p - y)) + (self.reg / n_samples) * self.w
            db = (1 / n_samples) * np.sum(p - y)

            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        return self._sigmoid(np.dot(X, self.w) + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


## 🧪 Project 2: Credit Card Default Pipeline with Imbalance Mitigation
We employ `imblearn.SMOTE` to address rare positives (defaults) and tune boundaries using Precision-Recall.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
from imblearn.over_sampling import SMOTE

# Generate imbalanced dataset
np.random.seed(42)
X_sim = np.random.randn(1000, 5)
y_sim = np.random.choice([0, 1], size=1000, p=[0.95, 0.05])

# SMOTE imbalance solver
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_sim, y_sim)

X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]
print('ROC AUC Score:', roc_auc_score(y_test, probs))
